In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from getdist import plots, MCSamples
from getdist.mcsamples import loadMCSamples

### Setup and Configuration

In [ ]:
# Assuming that the notebook is in 'notebooks/' and data is in 'data/', if not change the path according to your data location
chain_dir = "../data/chains/GWHIcross" 
chain_prefix = "2026-04-20_300000_"
chain_root = os.path.join(chain_dir, chain_prefix)

plot_outdir = "../data/plots"

print(plot_outdir)
os.makedirs(plot_outdir, exist_ok=True)

# Injected / Fiducial Cosmological Values
injected_params = {
    'omega_cdm': 0.1188,
    'H0': 67.74,
    'Omega_m': 0.3085,
    'Omega_Lambda': 0.6915,
}

print(f"Looking for chains at: {chain_root}*")
print("Found files:", glob.glob(f"{chain_root}*.txt"))

### Load Chains and Extract Statistics

In [ ]:
# Load all chains together (automatically handled by GetDist)
samples = loadMCSamples(chain_root, settings={'combine': True, 'ignore_rows': 0.3})

print("\n--- Parameter List ---")
for p in samples.getParamNames().names:
    print(f"- {p.name} ({p.label})")

print("\n--- Marginalized Statistics ---")
marginals = samples.getMargeStats()
print(marginals)

### Corner Plot

In [ ]:
# Parameters to plot
params_to_plot = ['H0', 'Omega_m', 'Omega_Lambda']
theme_color = '#004c99' 

# Initialize GetDist Plotter
g = plots.get_subplot_plotter()

# Font and Style settings
g.settings.axes_fontsize = 14 
g.settings.lab_fontsize = 18
g.settings.legend_fontsize = 14 
g.settings.axes_labelsize = 14 
g.settings.title_limit_fontsize = 12 
g.settings.axis_marker_color = 'grey' 
g.settings.axis_marker_lw = 1.5 
g.settings.linewidth_contour = 2.5  
g.settings.linewidth = 3.0          

# Generate Triangle Plot
g.triangle_plot([samples], params_to_plot, 
                filled=False, 
                markers=injected_params, 
                colors=[theme_color], 
                contour_colors=[theme_color],
                title_limit=1)

# Apply thickened panel borders
fig = g.fig
for i in range(len(params_to_plot)):
    for j in range(len(params_to_plot)):
        ax = g.subplots[i, j]
        if ax is not None:
            for spine in ax.spines.values():
                spine.set_linewidth(1.5)

# Save and Show
save_path = os.path.join(plot_outdir, 'corner_plot.png')
plt.savefig(save_path, dpi=400, bbox_inches='tight', transparent=False)
plt.show()

### Chain Convergence

In [ ]:
# Automatically find all chain files matching the prefix
chain_files = sorted(glob.glob(f"{chain_root}*[0-9].txt"))
chains = [np.loadtxt(f) for f in chain_files]

def plot_trace(chains, param_index, param_name, injected_val):
    plt.figure(figsize=(12, 5))
    for i, c in enumerate(chains, start=1):
        plt.plot(c[:, param_index], label=f"Chain {i}", alpha=0.7)
    
    plt.axhline(injected_val, color="black", linestyle="--", label="Injected value")
    plt.xlabel("Iteration")
    plt.ylabel(param_name)
    plt.title(f"MCMC Trace: {param_name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

# H0 is typically column 3, Omega_cdm is column 2 (verify with your .paramnames!)
plot_trace(chains, param_index=3, param_name="H0", injected_val=injected_params['H0'])
plot_trace(chains, param_index=2, param_name="Omega_cdm", injected_val=injected_params['omega_cdm'])